In [50]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# The sentences to encode
sentences = [
    "dog",
    "cat",
    "math"
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)

(3, 384)
tensor([[0.99999982, 0.66063738, 0.32408801],
        [0.66063738, 1.00000012, 0.32070020],
        [0.32408804, 0.32070023, 1.00000000]])


In [61]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

word1, word2, template = "apple", "google", "i work at {}"

def get_embedding(inputs, model):
    with torch.no_grad():
        output = model(**inputs)
    token_vecs = output.last_hidden_state[0]
    mask = inputs['attention_mask'][0].unsqueeze(-1).float()
    mean_vec = (token_vecs * mask).sum(0) / mask.sum()
    normalized = F.normalize(mean_vec.unsqueeze(0), p=2, dim=1)
    return normalized

# 单独的词
in1 = tokenizer(word1, return_tensors='pt')
in2 = tokenizer(word2, return_tensors='pt')

# 句子里的词
in1_sent = tokenizer(template.format(word1), return_tensors='pt')
in2_sent = tokenizer(template.format(word2), return_tensors='pt')

vec1_alone = get_embedding(in1, model)
vec2_alone = get_embedding(in2, model)
vec1_sent  = get_embedding(in1_sent, model)
vec2_sent  = get_embedding(in2_sent, model)

sim_alone = F.cosine_similarity(vec1_alone, vec2_alone).item()
sim_sent  = F.cosine_similarity(vec1_sent,  vec2_sent).item()

print(f"apple vs google (alone):   {sim_alone:.8f}")
print(f"apple vs google (context): {sim_sent:.8f}")
print(f"variation: {'↑ closer' if sim_sent > sim_alone else '↓ further'}")

apple vs google (alone):   0.32428783
apple vs google (context): 0.56958693
variation: ↑ closer


In [51]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# The sentences to encode
sentences = [
    "google",
    "apple",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)

(2, 384)
tensor([[1.00000024, 0.32428777],
        [0.32428777, 1.00000072]])


In [47]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

# 单独的词
inputs_dog = tokenizer("dog", return_tensors='pt')
inputs_cat = tokenizer("cat", return_tensors='pt')

inputs_dog_sent = tokenizer("this is a dog", return_tensors='pt')
inputs_cat_sent = tokenizer("this is a cat", return_tensors='pt')

with torch.no_grad():
    out_dog = model(**inputs_dog)
    out_cat = model(**inputs_cat)
    out_dog_sent = model(**inputs_dog_sent)
    out_cat_sent = model(**inputs_cat_sent)

tokens_dog_sent = tokenizer.convert_ids_to_tokens(inputs_dog_sent['input_ids'][0])
tokens_cat_sent = tokenizer.convert_ids_to_tokens(inputs_cat_sent['input_ids'][0])

dog_idx = tokens_dog_sent.index('dog')
cat_idx = tokens_cat_sent.index('cat')

vec_dog_alone = out_dog.last_hidden_state[0, 1, :]       
vec_cat_alone = out_cat.last_hidden_state[0, 1, :]       
vec_dog_sent  = out_dog_sent.last_hidden_state[0, dog_idx, :]  
vec_cat_sent  = out_cat_sent.last_hidden_state[0, cat_idx, :]  

sim_alone = F.cosine_similarity(vec_dog_alone.unsqueeze(0), vec_cat_alone.unsqueeze(0))
sim_sent  = F.cosine_similarity(vec_dog_sent.unsqueeze(0),  vec_cat_sent.unsqueeze(0))

print("dog (alone) vs cat (alone):", sim_alone.item())
print("dog (context) vs cat (context):", sim_sent.item())

dog (alone) vs cat (alone): 0.5350431799888611
dog (context) vs cat (context): 0.5033237934112549


[What is all-MiniLM-L6-v2 model?](https://www.educative.io/answers/what-is-all-minilm-l6-v2-model)

[Semantic Textual Similarity](https://sbert.net/docs/sentence_transformer/usage/semantic_textual_similarity.html)

[hugging face all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) 

In [17]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

token_id = tokenizer.convert_tokens_to_ids("this")
print("Token ID:", token_id)

embedding_table = model.embeddings.word_embeddings.weight
this_embedding = embedding_table[token_id]

print("vector dimesion:", this_embedding.shape) 
print("the first 5 elements:", this_embedding[:5].detach().numpy())

Token ID: 2023
vector dimesion: torch.Size([384])
the first 5 elements: [-0.00191669 -0.00740279 -0.02672139  0.02120891  0.08740292]


In [19]:
token_id_is = tokenizer.convert_tokens_to_ids("is")
print("Token ID:", token_id_is)

embedding_table = model.embeddings.word_embeddings.weight
this_embedding = embedding_table[token_id_is]

print("vector dimesion:", this_embedding.shape) 
print("the first 5 elements:", this_embedding[:5].detach().numpy())

Token ID: 2003
vector dimesion: torch.Size([384])
the first 5 elements: [-0.02077937 -0.02789463 -0.0515293   0.03763008 -0.03176742]


In [20]:
token_id_a = tokenizer.convert_tokens_to_ids("a")
print("Token ID:", token_id_a)

embedding_table = model.embeddings.word_embeddings.weight
this_embedding = embedding_table[token_id_a]

print("vector dimesion:", this_embedding.shape) 
print("the first 5 elements:", this_embedding[:5].detach().numpy())

Token ID: 1037
vector dimesion: torch.Size([384])
the first 5 elements: [-0.0297142   0.00833924 -0.01707635  0.01374365 -0.0349828 ]


In [ ]:
token_id_dog = tokenizer.convert_tokens_to_ids("dog")
print("Token ID:", token_id_dog)

embedding_table = model.embeddings.word_embeddings.weight
this_embedding = embedding_table[token_id_dog]

print("vector dimesion:", this_embedding.shape) 
print("the first 5 elements:", this_embedding[:5].detach().numpy())

Token ID: 3899
vector dimesion: torch.Size([384])
the first 5 elements: [ 0.02136218  0.011649    0.07449205 -0.11418393 -0.07305472]


In [35]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

sent1 = "This is a dog"

# Tokenize
inputs1 = tokenizer(sent1, return_tensors='pt')

with torch.no_grad():
    output1 = model(**inputs1)

tokens1 = tokenizer.convert_ids_to_tokens(inputs1['input_ids'][0])

this_idx1 = tokens1.index('this')
this_vec1 = output1.last_hidden_state[0, this_idx1, :5]
this_id = tokenizer.convert_tokens_to_ids('this')
static_vec_this = model.embeddings.word_embeddings.weight[this_id, :5]

print("the first 5 elements before adjustment of 'this':", static_vec_this.detach().numpy())
print("the first 5 elements after adjustment of 'this':", this_vec1.numpy())

is_idx1 = tokens1.index('is')
is_vec1 = output1.last_hidden_state[0, is_idx1, :5]
is_id = tokenizer.convert_tokens_to_ids('is')
static_vec_is = model.embeddings.word_embeddings.weight[is_id, :5]

print("the first 5 elements before adjustment of 'is':", static_vec_is.detach().numpy())
print("the first 5 elements after adjustment of 'is':", is_vec1.numpy())

a_idx1 = tokens1.index('a')
a_vec1 = output1.last_hidden_state[0, a_idx1, :5]
a_id = tokenizer.convert_tokens_to_ids('a')
static_vec_a = model.embeddings.word_embeddings.weight[a_id, :5]

print("the first 5 elements before adjustment of 'a':", static_vec_a.detach().numpy())
print("the first 5 elements after adjustment of 'a':", a_vec1.numpy())

dog_idx1 = tokens1.index('dog')
dog_vec1 = output1.last_hidden_state[0, dog_idx1, :5]
dog_id = tokenizer.convert_tokens_to_ids('dog')
static_vec_dog = model.embeddings.word_embeddings.weight[dog_id, :5]

print("the first 5 elements before adjustment of 'dog':", static_vec_dog.detach().numpy())
print("the first 5 elements after adjustment of 'dog':", dog_vec1.numpy())

the first 5 elements before adjustment of 'this': [-0.00191669 -0.00740279 -0.02672139  0.02120891  0.08740292]
the first 5 elements after adjustment of 'this': [-0.19542743  0.5070529   0.02721323  0.06611808  0.2429709 ]
the first 5 elements before adjustment of 'is': [-0.02077937 -0.02789463 -0.0515293   0.03763008 -0.03176742]
the first 5 elements after adjustment of 'is': [ 0.00139954  0.66500556  0.14664428  0.11904284 -0.33281806]
the first 5 elements before adjustment of 'a': [-0.0297142   0.00833924 -0.01707635  0.01374365 -0.0349828 ]
the first 5 elements after adjustment of 'a': [-0.5436946   0.2730863   0.01969518  0.17753628 -0.56696236]
the first 5 elements before adjustment of 'dog': [ 0.02136218  0.011649    0.07449205 -0.11418393 -0.07305472]
the first 5 elements after adjustment of 'dog': [-0.00604768 -0.2981155   0.279578    0.4800337  -1.0622689 ]


In [41]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

all_vecs_full = output1.last_hidden_state[0, :, :]  
manual_mean_full = all_vecs_full.mean(dim=0)
normalized_full = F.normalize(manual_mean_full.unsqueeze(0), p=2, dim=1)

print("manual mean (前5):", manual_mean_full[:5].numpy())      
print("normalized (前5):", normalized_full[0, :5].numpy())      

manual mean (前5): [-0.15300345  0.2339977   0.12918921  0.20332403 -0.39959696]
normalized (前5): [-0.02895953  0.04428961  0.02445212  0.03848389 -0.07563319]


In [43]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Sentences we want sentence embeddings for
sentences = ['This is a dog']

# Load model from HuggingFace Hub
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

# Tokenize sentences
encoded_input = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

# Compute token embeddings
with torch.no_grad():
    model_output = model(**encoded_input)

# Perform pooling
sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

# Normalize embeddings
sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

torch.set_printoptions(sci_mode=False, precision=8)

print("Sentence embeddings:")
print(sentence_embeddings.shape)
print(sentence_embeddings)

Sentence embeddings:
torch.Size([1, 384])
tensor([[    -0.02895953,      0.04428961,      0.02445212,      0.03848389,
             -0.07563319,     -0.00766353,      0.02784445,     -0.05068679,
              0.08903737,      0.03030875,     -0.03078724,     -0.03723929,
             -0.00466351,      0.03979236,     -0.04628047,     -0.01512911,
             -0.04887405,     -0.01469521,      0.00862353,     -0.05302022,
             -0.05279782,      0.09523399,     -0.05358823,     -0.00201563,
             -0.06958959,      0.04390144,      0.02495688,     -0.06626028,
              0.03341731,      0.01712984,     -0.01597504,      0.00045907,
              0.01256405,      0.02987900,     -0.03686533,     -0.04216761,
              0.01911844,      0.02883479,      0.07049067,      0.07668872,
              0.04388579,     -0.05737624,      0.01097914,     -0.03398279,
             -0.04261504,      0.05565324,     -0.07959912,     -0.05882544,
              0.04381804,     -0.0